# HomeReach Melbourne — 08: Compare transport supply and rental costs

**Stage:** descriptive exploration of the completed pipeline outputs. No scoring, recommendations, forecasts or new source downloads.

We will produce a **31-row, one-LGA-per-row comparison table** and four interactive charts:
1. Morning departure-event totals split by service category.
2. Departure events versus distinct trips: why they are not the same count.
3. Two explicit stop-record denominators: all assigned records versus records with counted morning service.
4. Published rent versus morning distinct trips, with both source dates shown.

| Source component | Period / scope |
|---|---|
| Rental observations | **2025Q3**, `2 bedroom flats`; 31 source-group metropolitan LGAs |
| Boundary identifiers | **ABS LGA 2025**; the previously reviewed crosswalk |
| Transport | **2026-10-13**, Australia/Melbourne service-day time; feeds 2, 3 and 4 |
| Morning window | **07:00 inclusive to 09:00 exclusive** |

**This is a comparison of differently dated snapshots, not a same-period causal study.** `quarter` in the joined table remains the rental quarter; `transport_service_date` is a separate field. Do not copy the new transport measures across the historical rental panel for model evaluation.

Use your existing **homereach** kernel. Run from top to bottom using notebook controls. This notebook reads small CSV/JSON outputs; it **never opens the raw GTFS ZIP or scans stop_times.txt**. No earlier notebook needs to remain running.

Generated tables/charts go only to `data/interim/transport_rent_eda/`; the generated report is `docs/transport_rent_eda.md`. Existing source files, processed tables, configuration, registry and app are not edited.

## 1. Imports, paths and small validation helpers

We reuse the rental CSV reader because CSV does not retain data types by itself. All identifiers in other tables start as strings. Numeric count columns will be explicitly converted and checked.

A fingerprint verifies consistency with a local pipeline report; it does not independently certify the publisher's data. Missing or changed inputs must be investigated rather than silently rebuilt by this notebook.

In [1]:
print("Starting notebook 08 — transport and rental comparison...", flush=True)
from pathlib import Path
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
from IPython.display import display
from homereach.io.dffh_rent import COLUMN_ORDER, KEY_COLUMNS, read_panel_csv, load_snapshot


def require(condition, message):
    if not condition:
        raise ValueError(message)


def fingerprint(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def read_json(path):
    value = json.loads(Path(path).read_text(encoding="utf-8-sig"))
    require(isinstance(value, dict), f"Expected a JSON object: {path}")
    return value


def require_key(frame, columns, label):
    require(set(columns).issubset(frame.columns), f"{label}: missing key columns.")
    require(frame[columns].notna().all().all(), f"{label}: null key.")
    require(all(frame[c].astype("string").str.strip().ne("").all() for c in columns), f"{label}: empty key.")
    require(not frame.duplicated(columns).any(), f"{label}: duplicate key {columns}.")


def count_columns(frame, columns, label):
    frame = frame.copy()
    for column in columns:
        require(column in frame, f"{label}: missing {column}.")
        value = pd.to_numeric(frame[column], errors="raise")
        array = value.to_numpy(dtype=float)
        require(np.isfinite(array).all() and (array >= 0).all() and (array % 1 == 0).all(),
                f"{label}: {column} must contain finite, non-negative whole counts.")
        frame[column] = value.astype("int64")
    return frame


def complete_one_to_one(left, right, key, label):
    require_key(left, [key], f"{label} left")
    require_key(right, [key], f"{label} right")
    require(set(left[key]) == set(right[key]), f"{label}: different or missing keys; do not fill missing matches with zero.")
    overlap = (set(left.columns) & set(right.columns)) - {key}
    require(not overlap, f"{label}: unexpected overlapping columns {sorted(overlap)}.")
    result = left.merge(right, on=key, how="left", validate="one_to_one", indicator="_join_status")
    require(result["_join_status"].eq("both").all(), f"{label}: unmatched rows.")
    return result.drop(columns="_join_status")


def safe_ratio(numerator, denominator):
    # Undefined denominators stay missing; they are not converted to zero.
    return numerator.astype(float).div(denominator.astype(float).where(denominator.gt(0)))


ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "pyproject.toml").is_file() and (p / "data").is_dir()), None)
require(ROOT is not None, "Open this notebook inside your existing HomeReach project.")
TRANSPORT = ROOT / "data/processed/transport"
REPORT_PATH = TRANSPORT / "transport_pipeline_report.json"
GEO_REPORT_PATH = ROOT / "data/interim/geography/geography_pipeline_report.json"
CROSSWALK_PATH = ROOT / "data/interim/geography/lga_crosswalk_2025.csv"
RENT_PATH = ROOT / "data/processed/rent_quarterly.csv"
RENT_REPORT_PATH = ROOT / "data/processed/rent_quality_report.json"
SPEC_PATH = ROOT / "config/gtfs_snapshot.json"
RENT_SPEC_PATH = ROOT / "config/rent_snapshot.json"
GEO_SPEC_PATH = ROOT / "config/lga_snapshot.json"
ALIAS_PATH = ROOT / "config/lga_name_aliases_2025.csv"
AM_WINDOW = "am_peak_0700_0900"
DAY_WINDOW = "service_day"
CODE = "lga_code_2025"
SERVICE_DATE = "2026-10-13"
TIMEZONE = "Australia/Melbourne"
RENT_QUARTER = "2025Q3"
CATEGORY_FEED = {
    "train_not_labelled_replacement": "2", "labelled_replacement_bus": "2",
    "tram": "3", "myki_bus": "4",
}
CATEGORY_LABELS = {
    "train_not_labelled_replacement": "Train feed (not labelled replacement)",
    "labelled_replacement_bus": "Labelled replacement bus", "tram": "Tram", "myki_bus": "Myki bus",
}
COUNT_FIELDS = ["eligible_departure_events", "unique_trip_records", "served_stop_records",
                "unique_route_records", "near_boundary_events", "extended_time_events"]
GROUP = [CODE, "feed_id", "service_category", "window"]
base_paths = [REPORT_PATH, GEO_REPORT_PATH, CROSSWALK_PATH, RENT_PATH, RENT_REPORT_PATH,
              SPEC_PATH, RENT_SPEC_PATH, GEO_SPEC_PATH, ALIAS_PATH]
for path in base_paths:
    require(path.is_file(), f"Missing {path.relative_to(ROOT)}. Finish the relevant existing pipeline first.")
input_hashes = {path: fingerprint(path) for path in base_paths}
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 35)
print("Python:", sys.executable)
print("pandas:", pd.__version__, "| Plotly:", plotly.__version__)
print("Project root:", ROOT)


Starting notebook 08 — transport and rental comparison...
Python: c:\Users\PhamDang\anaconda3\envs\homereach\python.exe
pandas: 3.0.3 | Plotly: 7.1.0
Project root: D:\My project\homereach_melbourne


## 2. Verify the reports and load only processed tables

The transport report stores output hashes relative to its own directory; the geography report stores paths relative to the project. Both slash conventions are handled for Windows compatibility.

We do not compare every old documentation hash: `--update-registry` intentionally changes the registry after a build. We instead verify the data/configuration needed for this analysis and the source fingerprints tying the pipelines together.

In [2]:
transport_report = read_json(REPORT_PATH)
geo_report = read_json(GEO_REPORT_PATH)
rent_report = read_json(RENT_REPORT_PATH)
spec = read_json(SPEC_PATH)
rent_spec = load_snapshot(RENT_SPEC_PATH)
geo_spec = read_json(GEO_SPEC_PATH)
require(transport_report.get("builder") == "scripts/build_transport.py" and transport_report.get("check_only") is False,
        "A completed transport build report is required.")
for flag in ["counts_reconciled", "csv_readback_matches", "gpkg_readback_matches", "upstream_inputs_unchanged"]:
    require(transport_report.get(flag) is True, f"Transport report is incomplete: {flag}.")
require(transport_report.get("transport_snapshot") == spec, "Transport configuration differs from its report.")
require(spec["service_date"] == SERVICE_DATE and spec["timezone"] == TIMEZONE,
        "This notebook is reviewed for 2026-10-13 in Melbourne service-day time.")
require(spec["category_feed"] == CATEGORY_FEED and spec["windows"][AM_WINDOW] == [25200, 32400],
        "Reviewed categories or morning window changed.")
require(spec["rental_reporting_quarter"] == RENT_QUARTER and spec["lga_count"] == 31,
        "Reviewed rental period or study area changed.")
require(geo_report.get("boundary_snapshot") == geo_spec and geo_report.get("check_only") is False,
        "A compatible completed geography build is required.")
require(geo_spec["map_quarter"] == RENT_QUARTER and geo_spec["boundary_year"] == 2025,
        "Geography/rental snapshot mismatch.")
require(rent_report.get("source", {}).get("sha256") == rent_spec["sha256"], "Rental source snapshot mismatch.")
require(rent_report.get("output_sha256", {}).get(RENT_PATH.name) == fingerprint(RENT_PATH),
        "Rental CSV differs from its own quality report.")
geo_hashes = {k.replace("\\", "/"): v for k, v in geo_report["output_sha256"].items()}
require(geo_hashes.get(CROSSWALK_PATH.relative_to(ROOT).as_posix()) == fingerprint(CROSSWALK_PATH),
        "Crosswalk differs from the geography build report.")
# Native 2025 geometry is not needed for charting but its fingerprint links the geographic builds.
native_path = ROOT / "data/processed/geography/metro_lga_2025.gpkg"
require(native_path.is_file(), "Missing previously validated LGA geometry.")
input_hashes[native_path] = fingerprint(native_path)
transport_inputs = {k.replace("\\", "/"): v for k, v in transport_report["input_sha256"].items()}
for path in [CROSSWALK_PATH, native_path]:
    require(transport_inputs.get(path.relative_to(ROOT).as_posix()) == fingerprint(path),
            f"Transport was built from different geography: {path.name}.")
for filename, expected_hash in [("data/raw/transport/gtfs.zip", spec["sha256"]),
                                ("data/raw/geography/" + geo_spec["filename"], geo_spec["sha256"])]:
    require(transport_inputs.get(filename) == expected_hash, "Recorded source lineage differs.")


def load_transport_csv(relative):
    path = TRANSPORT / relative
    require(path.is_file(), f"Missing processed transport output: {relative}")
    digest = fingerprint(path)
    hashes = {k.replace("\\", "/"): v for k, v in transport_report["output_sha256"].items()}
    require(hashes.get(relative) == digest, f"{relative}: fingerprint differs from its completed pipeline report.")
    frame = pd.read_csv(path, dtype="string", keep_default_na=False)
    schema = transport_report["output_schemas"].get(relative, {})
    require(schema.get("rows") == len(frame) and schema.get("columns") == frame.columns.tolist(),
            f"{relative}: recorded schema differs.")
    input_hashes[path] = digest
    return frame


lga_service = count_columns(load_transport_csv("service/lga_service_indicators.csv"),
                            COUNT_FIELDS + ["eligible_events_excluding_near_boundary"], "LGA service")
stop_service = count_columns(load_transport_csv("service/stop_service_indicators.csv"),
                             ["eligible_departure_events", "unique_trip_records", "unique_route_records",
                              "first_departure_seconds", "last_departure_seconds"], "stop service")
memberships = load_transport_csv("service/trip_lga_memberships.csv")
network_service = count_columns(load_transport_csv("service/feed_service_summary.csv"), COUNT_FIELDS, "study area")
locations = load_transport_csv("spatial/stop_lga_matches.csv")
crosswalk = pd.read_csv(CROSSWALK_PATH, dtype="string", keep_default_na=False)
panel = read_panel_csv(RENT_PATH)
require(panel.columns.tolist() == COLUMN_ORDER, "Rental 14-column contract changed.")
require_key(panel, KEY_COLUMNS, "rental panel")
require(len(panel) == 3286 and len(crosswalk) == 31, "Unexpected input scope.")
require_key(crosswalk, [CODE], "crosswalk codes")
require_key(crosswalk, ["rental_area_name"], "crosswalk names")
require(crosswalk["boundary_year"].eq("2025").all(), "Crosswalk boundary year changed.")
print("INPUT CHECKPOINT PASSED")
print("Transport indicator rows:", len(lga_service))
print("Rental rows:", len(panel), "| Crosswalk LGAs:", len(crosswalk))
print("Processed inputs match their local pipeline reports: True")
print("Raw GTFS reads or scans performed: 0")


INPUT CHECKPOINT PASSED
Transport indicator rows: 248
Rental rows: 3286 | Crosswalk LGAs: 31
Processed inputs match their local pipeline reports: True
Raw GTFS reads or scans performed: 0


## 3. Reconcile identities and totals before plotting

A trip can stop in several LGAs. Its **departure events** can be summed across uniquely assigned LGAs, but its **distinct-trip count cannot**. We check LGA counts against the trip/LGA membership table and compute study-area trips by deduplicating the feed-prefixed keys.

Similarly, a stop record can appear in more than one service category. We count distinct `stop_key` values rather than blindly summing category-level served-stop counts.

In [3]:
require_key(lga_service, GROUP, "LGA service grid")
require_key(stop_service, GROUP + ["stop_key"], "stop/category grid")
require_key(memberships, GROUP + ["trip_key"], "trip memberships")
require_key(network_service, ["feed_id", "service_category", "window"], "study-area summary")
require_key(locations, ["feed_id", "stop_id"], "stored locations")
require_key(locations, ["stop_key"], "stored stop keys")
require(locations.stop_key.eq(locations.feed_id + ":" + locations.stop_id).all(), "Location namespace changed.")
known_codes = set(crosswalk[CODE])
for label, frame in [("LGA", lga_service), ("stop", stop_service), ("membership", memberships), ("study area", network_service)]:
    require(frame.service_date.eq(SERVICE_DATE).all() and frame.agency_timezone.eq(TIMEZONE).all(), f"{label}: mixed dates/timezones.")
    require(frame.window.isin([AM_WINDOW, DAY_WINDOW]).all(), f"{label}: unexpected window.")
    require(frame.service_category.isin(CATEGORY_FEED).all(), f"{label}: unknown category.")
    require(frame.feed_id.eq(frame.service_category.map(CATEGORY_FEED)).all(), f"{label}: category/feed mismatch.")
    if CODE in frame:
        require(set(frame[CODE]).issubset(known_codes), f"{label}: unreviewed LGA code.")
expected_grid = {(c, f, cat, w) for c in known_codes for cat, f in CATEGORY_FEED.items()
                 for w in [AM_WINDOW, DAY_WINDOW]}
require(set(lga_service[GROUP].itertuples(index=False, name=None)) == expected_grid and len(lga_service) == 248,
        "Incomplete LGA/category/window grid. Missing rows are not assumed to be zero.")
identity = lga_service[[CODE, "abs_lga_name", "rental_area_name"]].drop_duplicates()
require_key(identity, [CODE], "service LGA identity")
pd.testing.assert_frame_equal(identity.sort_values(CODE).reset_index(drop=True),
                              crosswalk[[CODE, "abs_lga_name", "rental_area_name"]].sort_values(CODE).reset_index(drop=True),
                              check_dtype=False, check_exact=True)
require(memberships.groupby("trip_key")[["feed_id", "service_category", "route_key"]].nunique().eq(1).all().all(),
        "One trip belongs to conflicting feeds/categories/routes.")
require(memberships.trip_key.str.split(":", n=1).str[0].eq(memberships.feed_id).all(), "Trip namespace lost.")
require(stop_service.stop_key.eq(stop_service.feed_id + ":" + stop_service.stop_id).all(), "Service-stop namespace lost.")
assigned = locations.loc[locations.spatial_match_status.eq("matched_within"),
                         ["feed_id", "stop_id", "stop_key", "stop_name", CODE]].copy()
require(len(assigned) == 21426 and set(assigned[CODE]).issubset(known_codes), "Assigned stop inventory changed.")
stop_pairs = stop_service[["stop_key", CODE]].drop_duplicates()
require_key(stop_pairs, ["stop_key"], "service stop assignments")
assignment_check = stop_pairs.merge(assigned[["stop_key", CODE]], on="stop_key", how="left",
                                   suffixes=("_service", "_location"), validate="one_to_one", indicator=True)
require(assignment_check["_merge"].eq("both").all()
        and assignment_check[f"{CODE}_service"].eq(assignment_check[f"{CODE}_location"]).all(),
        "A served stop is missing or assigned to the wrong LGA.")
ref = lga_service.set_index(GROUP).sort_index()
for field, actual in [
    ("eligible_departure_events", stop_service.groupby(GROUP).eligible_departure_events.sum()),
    ("served_stop_records", stop_service.groupby(GROUP).stop_key.nunique()),
    ("unique_trip_records", memberships.groupby(GROUP).trip_key.nunique()),
    ("unique_route_records", memberships.groupby(GROUP).route_key.nunique()),
]:
    require(np.array_equal(actual.reindex(ref.index, fill_value=0).to_numpy(), ref[field].to_numpy()),
            f"Independent LGA count mismatch: {field}.")
net_keys = ["feed_id", "service_category", "window"]
net_ref = network_service.set_index(net_keys).sort_index()
for field, actual in [
    ("eligible_departure_events", lga_service.groupby(net_keys).eligible_departure_events.sum()),
    ("unique_trip_records", memberships.groupby(net_keys).trip_key.nunique()),
    ("unique_route_records", memberships.groupby(net_keys).route_key.nunique()),
    ("served_stop_records", stop_service.groupby(net_keys).stop_key.nunique()),
]:
    require(np.array_equal(actual.reindex(net_ref.index, fill_value=0).to_numpy(), net_ref[field].to_numpy()),
            f"Independent study-area mismatch: {field}.")
require(lga_service.eligible_events_excluding_near_boundary.eq(
    lga_service.eligible_departure_events - lga_service.near_boundary_events).all(), "Near-boundary subtraction mismatch.")
for column in COUNT_FIELDS[1:]:
    require(lga_service[column].le(lga_service.eligible_departure_events).all(), f"Impossible {column} count.")
day = lga_service.loc[lga_service.window.eq(DAY_WINDOW)].set_index([CODE, "service_category"]).sort_index()
am = lga_service.loc[lga_service.window.eq(AM_WINDOW)].set_index([CODE, "service_category"]).sort_index()
require(am[COUNT_FIELDS].le(day[COUNT_FIELDS]).all().all() and am.extended_time_events.eq(0).all(),
        "AM window is not a valid subset of the service day.")
AM_EVENTS = int(am.eligible_departure_events.sum())
AM_UNIQUE_TRIPS = int(memberships.loc[memberships.window.eq(AM_WINDOW), "trip_key"].nunique())
require(AM_EVENTS == 158932 and AM_UNIQUE_TRIPS == 5768, "Audited morning totals changed.")
print("ACCOUNTING CHECKPOINT PASSED")
print("AM counted departure events:", AM_EVENTS)
print("Distinct AM trips across the study area:", AM_UNIQUE_TRIPS)
print("LGA sum of distinct-trip counts (NOT a study-area trip total):", int(am.unique_trip_records.sum()))


ACCOUNTING CHECKPOINT PASSED
AM counted departure events: 158932
Distinct AM trips across the study area: 5768
LGA sum of distinct-trip counts (NOT a study-area trip total): 12546


## 4. Build one transport-comparison row per LGA

We combine category-level **events** by summing them. We derive LGA **trips**, **routes** and **served stop records** using `nunique()` on their own identifiers. The function includes the complete LGA inventory, so a verified absence of counted records becomes zero; a missing LGA input is rejected earlier.

The full service day includes extended times beyond 24:00. It is not a 24-hour calendar-day denominator and we do not divide it by 24 to claim an average frequency.

In [4]:
def aggregate_lgas(lga_table, trip_memberships, stop_table, codes, window):
    selected = lga_table.loc[lga_table.window.eq(window)]
    require(set(selected[CODE]) == set(codes), "Incomplete LGA input for aggregation.")
    trips = trip_memberships.loc[trip_memberships.window.eq(window)]
    stops = stop_table.loc[stop_table.window.eq(window)]
    result = pd.DataFrame({CODE: sorted(codes)}).set_index(CODE)
    result["departure_events"] = selected.groupby(CODE).eligible_departure_events.sum().reindex(result.index, fill_value=0)
    result["unique_trip_records"] = trips.groupby(CODE).trip_key.nunique().reindex(result.index, fill_value=0)
    result["served_stop_records"] = stops.groupby(CODE).stop_key.nunique().reindex(result.index, fill_value=0)
    result["unique_route_records"] = trips.groupby(CODE).route_key.nunique().reindex(result.index, fill_value=0)
    result["near_boundary_events"] = selected.groupby(CODE).near_boundary_events.sum().reindex(result.index, fill_value=0)
    result["extended_time_events"] = selected.groupby(CODE).extended_time_events.sum().reindex(result.index, fill_value=0)
    return result.astype("int64").reset_index()

am_totals = aggregate_lgas(lga_service, memberships, stop_service, known_codes, AM_WINDOW)
day_totals = aggregate_lgas(lga_service, memberships, stop_service, known_codes, DAY_WINDOW)
am_totals = am_totals.rename(columns={c: "am_" + c for c in am_totals if c != CODE})
day_totals = day_totals.rename(columns={c: "service_day_" + c for c in day_totals if c != CODE})
transport_comparison = complete_one_to_one(crosswalk[[CODE, "abs_lga_name", "rental_area_name", "source_region"]],
                                         am_totals, CODE, "AM totals")
transport_comparison = complete_one_to_one(transport_comparison, day_totals, CODE, "service-day totals")
transport_comparison["transport_service_date"] = SERVICE_DATE
transport_comparison["transport_timezone"] = TIMEZONE
transport_comparison["am_window"] = "07:00 inclusive to 09:00 exclusive"
transport_comparison["am_events_per_counted_trip"] = safe_ratio(
    transport_comparison.am_departure_events, transport_comparison.am_unique_trip_records)
transport_comparison["am_share_of_service_day_events_pct"] = 100 * safe_ratio(
    transport_comparison.am_departure_events, transport_comparison.service_day_departure_events)
transport_comparison["am_near_boundary_event_share_pct"] = 100 * safe_ratio(
    transport_comparison.am_near_boundary_events, transport_comparison.am_departure_events)
require(len(transport_comparison) == 31, "Expected one transport row per LGA.")
display(transport_comparison[["rental_area_name", "am_departure_events", "am_unique_trip_records",
                              "am_served_stop_records", "am_events_per_counted_trip"]]
        .sort_values(["am_departure_events", "rental_area_name"], ascending=[False, True]))


,rental_area_name,am_departure_events,am_unique_trip_records,am_served_stop_records,am_events_per_counted_trip
17,Melbourne,14096,1633,579,8.631966
20,Monash,7595,585,866,12.982906
26,Whitehorse,7426,558,888,13.308244
6,Darebin,7184,472,815,15.220339
2,Boroondara,6687,470,696,14.227660
5,Casey,6532,361,1290,18.094183
14,Manningham,6531,367,624,17.795640
19,Merri-bek,6469,490,782,13.202041
27,Whittlesea,6445,392,924,16.441327
11,Hume,6251,378,1031,16.537037


## 5. Make the denominator explicit

Two averages answer different questions:

- **Events / all assigned stop records:** includes assigned source records with no counted AM events.
- **Events / served stop records:** includes only records with at least one counted AM event.

Neither denominator is a count of residents, unique physical stations, trips or accessible locations. Duplicate physical locations across feeds remain separate; terminal/pickup rules can remove all counted events at a record. These are **descriptive sensitivity measures**, not validated accessibility scores or passenger waiting times.

A zero denominator is **undefined** and stays missing. A verified zero event count with a positive assigned-record denominator yields a real zero. The data below add zero events only after a validated assignment inventory is joined to the complete counted-stop summary.

In [5]:
am_stops = stop_service.loc[stop_service.window.eq(AM_WINDOW)]
am_stop_totals = am_stops.groupby([CODE, "stop_key"], as_index=False).eligible_departure_events.sum()
require_key(am_stop_totals, ["stop_key"], "AM counted stops")
stop_context = assigned.merge(am_stop_totals, on=[CODE, "stop_key"], how="left", validate="one_to_one", indicator=True)
# Earlier checks proved that all counted stops occur in the assigned inventory.
stop_context["am_events"] = stop_context.eligible_departure_events.fillna(0).astype("int64")
require(stop_context.loc[stop_context["_merge"].eq("both"), "am_events"].gt(0).all(), "A served stop has zero events.")
require(int(stop_context.am_events.sum()) == AM_EVENTS, "Stop-level denominator context lost events.")
stop_context = stop_context.drop(columns=["eligible_departure_events", "_merge"])
profile = stop_context.groupby(CODE).agg(
    assigned_stop_records=("stop_key", "size"),
    am_served_stop_records_check=("am_events", lambda s: int(s.gt(0).sum())),
    am_events_check=("am_events", "sum"),
    am_median_events_per_assigned_stop_record=("am_events", "median"),
).reindex(sorted(known_codes))
require(profile["assigned_stop_records"].notna().all(), "An LGA has no assignment inventory; investigate before assigning rates.")
for col in ["assigned_stop_records", "am_served_stop_records_check", "am_events_check"]:
    profile[col] = profile[col].astype("int64")
profile["assigned_records_without_counted_am_event"] = profile.assigned_stop_records - profile.am_served_stop_records_check
profile["mean_am_events_per_assigned_stop_record"] = safe_ratio(profile.am_events_check, profile.assigned_stop_records)
profile["mean_am_events_per_served_stop_record"] = safe_ratio(profile.am_events_check, profile.am_served_stop_records_check)
profile["assigned_record_share_with_counted_am_event_pct"] = 100 * safe_ratio(
    profile.am_served_stop_records_check, profile.assigned_stop_records)
denominator_comparison = complete_one_to_one(crosswalk[[CODE, "rental_area_name"]], profile.reset_index(), CODE, "denominator profile")
transport_comparison = complete_one_to_one(transport_comparison,
    profile.drop(columns=["am_served_stop_records_check", "am_events_check"]).reset_index(), CODE, "transport denominators")
check = transport_comparison.set_index(CODE).sort_index()
require(np.array_equal(check.am_departure_events, profile.am_events_check), "LGA denominator event totals disagree.")
require(np.array_equal(check.am_served_stop_records, profile.am_served_stop_records_check), "LGA served-record counts disagree.")
print("Assigned stop/platform records:", len(stop_context))
print("Records with a counted AM event:", int(stop_context.am_events.gt(0).sum()))
print("Assigned records without a counted AM event:", int(stop_context.am_events.eq(0).sum()))
display(denominator_comparison[["rental_area_name", "assigned_stop_records", "am_served_stop_records_check",
                                "mean_am_events_per_assigned_stop_record", "mean_am_events_per_served_stop_record"]].head(10))


Assigned stop/platform records: 21426
Records with a counted AM event: 20448
Assigned records without a counted AM event: 978


,rental_area_name,assigned_stop_records,am_served_stop_records_check,mean_am_events_per_assigned_stop_record,mean_am_events_per_served_stop_record
0,Banyule,589,579,8.514431,8.661485
1,Bayside,491,482,6.378819,6.497925
2,Boroondara,719,696,9.300417,9.607759
3,Brimbank,852,839,6.772300,6.877235
4,Cardinia,322,274,2.509317,2.948905
5,Casey,1366,1290,4.781845,5.063566
6,Darebin,876,815,8.200913,8.814724
7,Frankston,634,594,4.694006,5.010101
8,Glen Eira,672,653,7.571429,7.791730
9,Greater Dandenong,757,744,5.428005,5.522849


## 6. Join rents using the reviewed LGA codes

First use the existing reviewed crosswalk to attach a code to each latest-quarter rental row. Then join the transport comparison by **code**, not fuzzy names. Each table must have exactly one record per LGA, with no missing keys or lost areas.

All **14 original rental fields** are retained, including the original Count/Median Excel cell references. `quarter` and `quarter_end` describe rents; the transport date is explicitly separate. There is no attempt to claim that October 2026 service existed in September 2025.

The resulting table is for descriptive exploration. No combined score, winner, forecast, causal claim, regression or p-value is calculated.

In [6]:
expected_quarters = pd.period_range(rent_spec["first_quarter"], rent_spec["last_quarter"], freq="Q").astype(str).tolist()
for _, group in panel.groupby(["rental_area_name", "property_type"], sort=False):
    require(group.sort_values("quarter_end").quarter.tolist() == expected_quarters, "Rental quarter coverage changed.")
actual_dates = pd.DatetimeIndex(panel.quarter_end).as_unit("ns")
expected_dates = pd.PeriodIndex(panel.quarter, freq="Q").to_timestamp(how="end").normalize().as_unit("ns")
require(np.array_equal(actual_dates.asi8, expected_dates.asi8), "Rental dates and quarters disagree.")
latest = panel.loc[panel.quarter.eq(RENT_QUARTER)].copy().reset_index(drop=True)
require(len(latest) == 31 and latest.property_type.eq("2 bedroom flats").all(), "Unexpected rental period/category.")
require_key(latest, ["rental_area_name"], "latest rent")
require(latest.set_index("rental_area_name").source_region.to_dict()
        == crosswalk.set_index("rental_area_name").source_region.to_dict(), "Rental source groups differ from the crosswalk.")
latest_coded = complete_one_to_one(latest, crosswalk.drop(columns="source_region"), "rental_area_name", "rental code attachment")
transport_for_join = transport_comparison.drop(columns=["abs_lga_name", "rental_area_name", "source_region"])
combined = complete_one_to_one(latest_coded, transport_for_join, CODE, "rental/transport merge")
combined = combined.sort_values("rental_area_name", kind="stable").reset_index(drop=True)
require(len(combined) == 31 and combined[CODE].is_unique, "Joined table lost or multiplied LGAs.")
pd.testing.assert_frame_equal(combined[COLUMN_ORDER].sort_values(KEY_COLUMNS).reset_index(drop=True),
                              latest[COLUMN_ORDER].sort_values(KEY_COLUMNS).reset_index(drop=True), check_exact=True)
require(combined.quarter.eq(RENT_QUARTER).all() and combined.transport_service_date.eq(SERVICE_DATE).all(),
        "Source periods must remain separate.")
print("RENT–TRANSPORT JOIN CHECKPOINT PASSED")
print("Joined LGAs:", len(combined), "| Duplicate codes:", int(combined[CODE].duplicated().sum()))
print("Original rental columns preserved:", len(COLUMN_ORDER))
print("Rental quarter:", RENT_QUARTER, "| Transport service date:", SERVICE_DATE)
display(combined[["rental_area_name", CODE, "quarter", "median_weekly_rent", "published_count",
                  "transport_service_date", "am_departure_events", "am_unique_trip_records",
                  "mean_am_events_per_assigned_stop_record"]].head(10))


RENT–TRANSPORT JOIN CHECKPOINT PASSED
Joined LGAs: 31 | Duplicate codes: 0
Original rental columns preserved: 14
Rental quarter: 2025Q3 | Transport service date: 2026-10-13


,rental_area_name,lga_code_2025,quarter,median_weekly_rent,published_count,transport_service_date,am_departure_events,am_unique_trip_records,mean_am_events_per_assigned_stop_record
0,Banyule,20660,2025Q3,540.0,282,2026-10-13,5015,343,8.514431
1,Bayside,20910,2025Q3,650.0,210,2026-10-13,3132,236,6.378819
2,Boroondara,21110,2025Q3,600.0,542,2026-10-13,6687,470,9.300417
3,Brimbank,21180,2025Q3,400.0,224,2026-10-13,5770,429,6.772300
4,Cardinia,21450,2025Q3,450.0,24,2026-10-13,808,63,2.509317
5,Casey,21610,2025Q3,480.0,88,2026-10-13,6532,361,4.781845
6,Darebin,21890,2025Q3,530.0,473,2026-10-13,7184,472,8.200913
7,Frankston,22170,2025Q3,490.0,145,2026-10-13,2976,185,4.694006
8,Glen Eira,22310,2025Q3,600.0,760,2026-10-13,5088,465,7.571429
9,Greater Dandenong,22670,2025Q3,470.0,297,2026-10-13,4109,302,5.428005


## 7. Check the new analytical operations

These six self-check groups are separate from the 241 existing pytest tests. They deliberately test failures and denominator edge cases without altering your real data.

In the fictional trip example one trip visits two LGAs: **one distinct study-area trip**, not two. A shuffled copy must produce identical aggregates. The comparison join must reject duplicate or missing LGA keys.

In [7]:
def must_reject(action, label):
    try:
        action()
    except (ValueError, pd.errors.MergeError):
        return
    raise AssertionError(f"Expected rejection did not occur: {label}")

self_checks = []
left = pd.DataFrame({CODE: ["A", "B"], "rent": [500, 600]})
right = pd.DataFrame({CODE: ["A", "B"], "events": [10, 20]})
must_reject(lambda: complete_one_to_one(left, pd.concat([right, right.iloc[[0]]]), CODE, "fixture"), "duplicate key")
self_checks.append("duplicate-key rejection")
must_reject(lambda: complete_one_to_one(left, right.iloc[[0]], CODE, "fixture"), "missing code")
self_checks.append("missing-code rejection")
joined_example = complete_one_to_one(left, right.iloc[::-1], CODE, "fixture")
require(joined_example["events"].tolist() == [10, 20], "Join used row positions instead of codes.")
self_checks.append("code-based join independent of row order")
ratios = safe_ratio(pd.Series([10, 0, 0]), pd.Series([2, 2, 0]))
require(ratios.iloc[0] == 5 and ratios.iloc[1] == 0 and pd.isna(ratios.iloc[2]), "Zero-denominator semantics failed.")
self_checks.append("zero versus undefined ratio")
fixture_members = pd.DataFrame({CODE: ["A", "B"], "trip_key": ["2:T1", "2:T1"]})
require(fixture_members.groupby(CODE).trip_key.nunique().sum() == 2 and fixture_members.trip_key.nunique() == 1,
        "Cross-LGA trip deduplication example failed.")
self_checks.append("cross-LGA non-additivity")
shuffled = aggregate_lgas(lga_service.sample(frac=1, random_state=8), memberships.sample(frac=1, random_state=8),
                          stop_service.sample(frac=1, random_state=8), known_codes, AM_WINDOW)
original = aggregate_lgas(lga_service, memberships, stop_service, known_codes, AM_WINDOW)
pd.testing.assert_frame_equal(original, shuffled, check_exact=True)
self_checks.append("aggregation independent of row order")
print("Six analytical self-check groups passed:")
for label in self_checks:
    print(" -", label)


Six analytical self-check groups passed:
 - duplicate-key rejection
 - missing-code rejection
 - code-based join independent of row order
 - zero versus undefined ratio
 - cross-LGA non-additivity
 - aggregation independent of row order


## 8. Chart 1 — Morning events by LGA and category

A bar is one LGA; its segments are the four reviewed service categories. Length counts **eligible stop-time visits**, not separate vehicles. The bars are sorted by that count for readability, not by rental suitability.

Large LGAs or trips calling at many local stops can accumulate many events. Retain that structural limitation instead of labelling the longest bar "best public transport". Labelled replacement buses remain a separate series even though their AM counts are zero in this snapshot.

In [8]:
FIGURES = {}
transport_comparison = transport_comparison.sort_values("rental_area_name", kind="stable").reset_index(drop=True)
bar_order = transport_comparison.sort_values(["am_departure_events", "rental_area_name"], ascending=[False, True]).rental_area_name.tolist()
am_categories = lga_service.loc[lga_service.window.eq(AM_WINDOW)].copy()
am_categories["service_category_label"] = am_categories.service_category.map(CATEGORY_LABELS)


def format_chart(fig, title, subtitle, *, height=720, left=85):
    fig.update_layout(title={"text": title + "<br><sup>" + subtitle + "</sup>", "x": 0.02},
                      height=height, autosize=True, margin=dict(l=left, r=40, t=110, b=155),
                      font=dict(size=13), legend=dict(orientation="h", x=0, y=-0.16, xanchor="left", yanchor="top"))
    return fig


fig = go.Figure()
for category, label in CATEGORY_LABELS.items():
    subset = am_categories.loc[am_categories.service_category.eq(category)].set_index("rental_area_name").reindex(bar_order)
    require(subset[CODE].notna().all(), "Chart is missing a category/LGA row.")
    fig.add_trace(go.Bar(
        x=subset.eligible_departure_events.to_numpy(), y=subset.index.tolist(), orientation="h", name=label,
        customdata=subset[[CODE, "unique_trip_records", "served_stop_records", "near_boundary_events"]].to_numpy(),
        hovertemplate="%{y}<br>ABS code: %{customdata[0]}<br>Events: %{x:,}<br>Trips in this LGA/category: %{customdata[1]:,}"
                      "<br>Served stop records: %{customdata[2]:,}<br>Near-boundary events: %{customdata[3]:,}<extra>%{fullData.name}</extra>"))
format_chart(fig, "Morning scheduled departure events", "13 Oct 2026 · 07:00–09:00 Melbourne time · selected feeds and 31 LGAs", height=1180, left=190)
fig.update_layout(barmode="stack")
fig.update_yaxes(categoryorder="array", categoryarray=bar_order, autorange="reversed", title=None)
fig.update_xaxes(title="Eligible stop-time departure events (not distinct trips)", tickformat=",d", rangemode="tozero")
FIGURES["01_am_events_by_lga.html"] = fig
fig.show()


## 9. Chart 2 — Departure events are not the same as distinct trips

Each point is an LGA. The horizontal position shows AM departure events; the vertical position shows distinct trips contributing events there. The hover ratio is **counted stop visits per contributing trip inside that LGA**—not trips per hour or a route's frequency.

A trip touching several LGAs appears once in each relevant LGA's point, but only once in a distinct study-area trip total. We do not add a regression line or infer causality from these structurally related counts.

In [9]:
fig = go.Figure()
for region, subset in transport_comparison.groupby("source_region", sort=True):
    fig.add_trace(go.Scatter(
        x=subset.am_departure_events.to_numpy(), y=subset.am_unique_trip_records.to_numpy(), mode="markers", name=region,
        marker=dict(size=11),
        customdata=subset[["rental_area_name", CODE, "am_events_per_counted_trip", "am_served_stop_records"]].to_numpy(),
        hovertemplate="%{customdata[0]} (%{customdata[1]})<br>AM departure events: %{x:,}<br>Distinct AM trips in LGA: %{y:,}"
                      "<br>Events per contributing trip: %{customdata[2]:.2f}<br>Served records: %{customdata[3]:,}<extra></extra>"))
format_chart(fig, "One trip can contribute many departure events", "13 Oct 2026 · 07:00–09:00 · each dot is one LGA; colours show rental-source groups")
fig.update_xaxes(title="Eligible AM departure events", tickformat=",d", rangemode="tozero")
fig.update_yaxes(title="Distinct AM trip records serving the LGA", tickformat=",d", rangemode="tozero")
FIGURES["02_events_versus_distinct_trips.html"] = fig
fig.show()


## 10. Chart 3 — The denominator changes the interpretation

Both bars for an LGA use the same AM event total. One divides by **all assigned stop/platform records**, the other by **only records with a counted AM event**. Excluding zero-event records cannot lower the arithmetic average, but can hide how many assigned records have no counted event under our rules.

This is not a population weighting, network walkability analysis, mean passenger wait, or proof that a stop has no service. The time interval is always the two-hour window; these values are **events per stop record during the window**, not per minute.

In [10]:
rate_order = transport_comparison.sort_values(["mean_am_events_per_assigned_stop_record", "rental_area_name"],
                                              ascending=[False, True]).rental_area_name.tolist()
rate_data = transport_comparison.set_index("rental_area_name").reindex(rate_order)
fig = go.Figure()
for field, label in [
    ("mean_am_events_per_assigned_stop_record", "All assigned stop records (includes zeros)"),
    ("mean_am_events_per_served_stop_record", "Only records with counted AM events"),
]:
    fig.add_trace(go.Bar(
        x=rate_data[field].to_numpy(), y=rate_data.index.tolist(), orientation="h", name=label,
        customdata=rate_data[["assigned_stop_records", "am_served_stop_records", "assigned_records_without_counted_am_event",
                              "am_departure_events", "am_median_events_per_assigned_stop_record"]].to_numpy(),
        hovertemplate="%{y}<br>Mean: %{x:.2f}<br>All assigned records: %{customdata[0]:,}<br>Records with AM events: %{customdata[1]:,}"
                      "<br>Records with zero counted AM events: %{customdata[2]:,}<br>AM events: %{customdata[3]:,}"
                      "<br>Median events across all assigned records: %{customdata[4]:.1f}<extra>%{fullData.name}</extra>"))
format_chart(fig, "Two denominators for the same morning event total", "13 Oct 2026 · 07:00–09:00 · diagnostic averages, not accessibility scores", height=1350, left=190)
fig.update_layout(barmode="group")
fig.update_yaxes(categoryorder="array", categoryarray=rate_order, autorange="reversed", title=None)
fig.update_xaxes(title="Mean eligible AM departure events per stop/platform record", rangemode="tozero")
FIGURES["03_stop_record_denominators.html"] = fig
fig.show()


## 11. Chart 4 — Put the rental and transport snapshots together

Each point is one of the 31 matched LGAs. The horizontal axis is the **published 2025Q3 median weekly rent** for two-bedroom flats; the vertical axis is the **2026-10-13 distinct AM trip count** under our transport rules.

Moving left means a lower published rent; moving up means more counted distinct trips. Those are descriptive trade-offs—not a recommendation or an objective definition of a "good" LGA. A high LGA trip count need not give a resident a useful route to their destination.

The axes describe different source periods. We retain the original rent count and Excel cell reference in the hover. No trendline, inferential p-value, forecast, Pareto recommendation or transport score is added.

DFFH uses bond information as a proxy for new lettings and excludes some accommodation types; these are not live rental listings or the prices of the same fixed properties over time. See the publisher: https://www.dffh.vic.gov.au/publications/rental-report

In [11]:
fig = go.Figure()
for region, subset in combined.groupby("source_region", sort=True):
    fig.add_trace(go.Scatter(
        x=subset.median_weekly_rent.to_numpy(), y=subset.am_unique_trip_records.to_numpy(), mode="markers", name=region,
        marker=dict(size=12),
        customdata=subset[["rental_area_name", CODE, "quarter", "transport_service_date", "published_count",
                           "source_median_cell", "am_departure_events", "am_served_stop_records"]].to_numpy(),
        hovertemplate="%{customdata[0]} (%{customdata[1]})<br>Rent: $%{x:.0f}/week · %{customdata[2]}"
                      "<br>Distinct AM trips: %{y:,} · %{customdata[3]}<br>Published rental count: %{customdata[4]:,}"
                      "<br>Rent source: 2br Flat!%{customdata[5]}<br>AM events: %{customdata[6]:,}"
                      "<br>Served stop records: %{customdata[7]:,}<extra></extra>"))
format_chart(fig, "Rent and scheduled service: differently dated snapshots", "Rent: 2025Q3, 2-bedroom flats · Transport: 13 Oct 2026, 07:00–09:00 · not a recommendation", height=760)
fig.update_xaxes(title="Published median weekly rent (AUD/week) — 2025Q3", tickprefix="$", tickformat=",.0f")
fig.update_yaxes(title="Distinct AM trip records serving the LGA — 13 Oct 2026", tickformat=",d", rangemode="tozero")
FIGURES["04_rent_versus_am_trips.html"] = fig
fig.show()


## 12. Export checked analytical tables, charts and interpretation notes

The original pipeline outputs are read-only inputs to this notebook. The new comparison table is an **interim analytical output**, not a replacement for the rental panel or a production recommendation table.

Six CSV tables are written and reloaded with their declared data types. Four independent HTML charts contain their own Plotly JavaScript; no external basemap, API, spreadsheet export or browser service is needed. A manifest records source periods, input fingerprints, definitions and output hashes. The generated report is replaced on rerun; keep personal observations in a separate Markdown cell.

CSV/HTML/report replacements are per-file, not an all-or-nothing database transaction. Do not run simultaneous writers. The manifest is published last.

In [12]:
source_periods = pd.DataFrame([
    {"component": "rent", "period": RENT_QUARTER, "scope": "2 bedroom flats; 31 source-group metropolitan LGAs",
     "source_file": rent_spec["filename"], "source_sha256": rent_spec["sha256"], "publisher_page": rent_spec["publisher_page"]},
    {"component": "boundaries", "period": "2025", "scope": "ABS statistical LGA boundaries; reviewed code/name crosswalk",
     "source_file": geo_spec["filename"], "source_sha256": geo_spec["sha256"], "publisher_page": geo_spec.get("publisher_page", "https://www.abs.gov.au/")},
    {"component": "transport", "period": SERVICE_DATE, "scope": "Feeds 2,3,4; Melbourne service day; AM 07:00 inclusive–09:00 exclusive",
     "source_file": spec["filename"], "source_sha256": spec["sha256"], "publisher_page": spec["publisher_page"]},
])
reconciliation_rows = []
for w in [DAY_WINDOW, AM_WINDOW]:
    frame = memberships.loc[memberships.window.eq(w)]
    events = int(lga_service.loc[lga_service.window.eq(w), "eligible_departure_events"].sum())
    reconciliation_rows.append({"window": w, "service_date": SERVICE_DATE, "agency_timezone": TIMEZONE,
        "eligible_departure_events": events, "distinct_study_area_trips": int(frame.trip_key.nunique()),
        "sum_of_lga_trip_counts_not_a_study_area_total": int(frame.groupby(CODE).trip_key.nunique().sum()),
        "distinct_study_area_served_stop_records": int(stop_service.loc[stop_service.window.eq(w), "stop_key"].nunique())})
reconciliation = pd.DataFrame(reconciliation_rows)
TABLES = {
    "am_lga_category_comparison.csv": am_categories.sort_values([CODE, "service_category"]).reset_index(drop=True),
    "lga_transport_comparison.csv": transport_comparison,
    "rent_transport_comparison.csv": combined,
    "am_stop_denominator_comparison.csv": denominator_comparison.sort_values(CODE).reset_index(drop=True),
    "study_area_reconciliation.csv": reconciliation,
    "source_periods.csv": source_periods,
}
OUT = ROOT / "data/interim/transport_rent_eda"
SUMMARY_PATH = ROOT / "docs/transport_rent_eda.md"
OUT.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)


def check_inputs_unchanged():
    for path, digest in input_hashes.items():
        require(path.is_file() and fingerprint(path) == digest, f"Input changed during analysis: {path}")


def reload_csv_like(path, frame):
    dtype_map, date_cols = {}, []
    for col in frame:
        if pd.api.types.is_datetime64_any_dtype(frame[col].dtype):
            date_cols.append(col)
        elif pd.api.types.is_integer_dtype(frame[col].dtype):
            dtype_map[col] = "Int64" if str(frame[col].dtype) == "Int64" else "int64"
        elif pd.api.types.is_float_dtype(frame[col].dtype):
            dtype_map[col] = "float64"
        elif pd.api.types.is_bool_dtype(frame[col].dtype):
            dtype_map[col] = "bool"
        else:
            dtype_map[col] = "string"
    na = {c: [""] for c, dtype in dtype_map.items() if dtype == "float64"}
    back = pd.read_csv(path, dtype=dtype_map, parse_dates=date_cols, keep_default_na=False, na_values=na, float_precision="round_trip")
    for col in date_cols:
        back[col] = pd.DatetimeIndex(back[col]).as_unit("ns")
    pd.testing.assert_frame_equal(frame.reset_index(drop=True), back, check_dtype=False, check_exact=True)
    return back


largest_events = combined.sort_values(["am_departure_events", "rental_area_name"], ascending=[False, True]).iloc[0]
largest_trips = combined.sort_values(["am_unique_trip_records", "rental_area_name"], ascending=[False, True]).iloc[0]
lowest_rents = combined.loc[combined.median_weekly_rent.eq(combined.median_weekly_rent.min()), "rental_area_name"].tolist()
highest_rents = combined.loc[combined.median_weekly_rent.eq(combined.median_weekly_rent.max()), "rental_area_name"].tolist()
text_lines = [
    "# HomeReach Melbourne — Transport and rental snapshot comparison", "",
    f"Generated (UTC): {datetime.now(timezone.utc).isoformat()}", "", "## Scope", "",
    f"Rental period: **{RENT_QUARTER}**; 2 bedroom flats. Transport date: **{SERVICE_DATE}**, {TIMEZONE}, 07:00 inclusive–09:00 exclusive.",
    "The same 31 reviewed source-group metropolitan LGAs use the ABS 2025 identifiers. Feed scope remains 2,3,4; labelled replacements stay separate.",
    "This joins differently dated snapshots. No combined score, accessibility estimate, regression, prediction or recommendation is made.", "",
    "## Verified arithmetic", "",
    f"- AM eligible departure events: {AM_EVENTS:,}.",
    f"- Distinct AM trips across the study area: {AM_UNIQUE_TRIPS:,}.",
    f"- Sum of LGA-specific distinct AM trip counts: {int(am.unique_trip_records.sum()):,}; this is NOT the distinct study-area total.",
    f"- Assigned stop/platform records: {len(stop_context):,}; with counted AM events: {int(stop_context.am_events.gt(0).sum()):,}; with no counted AM event: {int(stop_context.am_events.eq(0).sum()):,}.",
    "- The 248-row service grid reconciles with the stop and trip-membership outputs. All 31 rental rows matched; all 14 original rental fields were preserved.",
    "- Six analytical self-check groups passed; these do not add to the project's pytest suite.", "",
    "## Descriptive observations from the supplied sources", "",
    f"- An LGA with the largest AM event count is {largest_events['rental_area_name']}: {int(largest_events['am_departure_events']):,} events, {int(largest_events['am_unique_trip_records']):,} distinct local contributing trips.",
    f"- An LGA with the largest distinct AM trip count is {largest_trips['rental_area_name']}: {int(largest_trips['am_unique_trip_records']):,} trips.",
    f"- Lowest published LGA median rent in this subset: ${combined.median_weekly_rent.min():.0f}/week ({', '.join(lowest_rents)}). Highest: ${combined.median_weekly_rent.max():.0f}/week ({', '.join(highest_rents)}).",
    "These are order statistics, not area recommendations. Ties in the comparison tables remain present.", "",
    "## How to read the four charts", "",
    "1. **Morning events by category:** a raw-volume comparison. More stop visits do not necessarily mean more useful routes or better residential access.",
    "2. **Events versus distinct trips:** the metrics differ because a trip calls at multiple stops and may visit multiple LGAs. No network trip total is obtained by adding LGA counts.",
    "3. **Two stop-record denominators:** compare events divided by all assigned records versus only served records. These means describe source records; they are not headways, passenger waits, physical-station counts or population-weighted access.",
    "4. **Rent versus morning trips:** horizontal rent values are 2025Q3, vertical service values are 2026-10-13. The scatter is not contemporaneous causal evidence or a forecast.", "",
    "## Cautions", "",
    "LGA extent, route arrangements and stop placement affect raw counts. Repeated/co-located records across feeds remain separate. Counts do not measure riders, vehicles, capacity, actual punctuality or commute time.",
    "A stop outside the polygons may serve residents inside; no walkshed or destination analysis has been performed. Near-boundary counts remain diagnostic, not error probabilities.",
    "Assigned locations with no counted AM event may be inactive, terminal-only, no-pickup, or affected by the selected service date and window. Zero does not prove absence of useful transport.",
    "Rent is an area-level nominal median based on the publisher's methodology, not a current listing, household-income affordability score, or price of a fixed property. Do not average LGA medians to manufacture a metropolitan median.",
    "Do not backfill the 2026 transport snapshot into historical rental model validation. A single weekday snapshot does not prove stable or disruption-free service.", "",
    "## Outputs", "",
    "Six CSV analysis tables and four self-contained HTML charts under data/interim/transport_rent_eda/. The manifest records definitions, schema and fingerprints.",
    "The input data, original notebooks, registry and Streamlit demo are not overwritten. Rerunning this final cell replaces this generated report; keep personal reflections separately.", "",
    "## Field definitions", "",
    "- am_departure_events: sum of eligible stop visits within one LGA in the two-hour morning window.",
    "- am_unique_trip_records / am_unique_route_records / am_served_stop_records: distinct feed-prefixed identities in that LGA/window.",
    "- am_events_per_counted_trip: events divided by the distinct contributing trip count; a stop-call ratio, not a frequency.",
    "- mean_am_events_per_assigned_stop_record: events divided by all assigned source stop/platform records, including zero-event records.",
    "- mean_am_events_per_served_stop_record: events divided by assigned records with at least one eligible AM event; undefined if none.",
    "- assigned_record_share_with_counted_am_event_pct: record-level fraction only, not the fraction of residents with transit access.",
    "- am_share_of_service_day_events_pct: AM event subset divided by full-service-day events; extended times are retained in the day total.",
    "- quarter / quarter_end / published_count / median_weekly_rent: the original rental fields. transport_service_date and transport_timezone are separate.", "",
    "## Primary references", "",
    "- https://www.dffh.vic.gov.au/publications/rental-report",
    "- https://gtfs.org/documentation/schedule/reference/",
    "- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html",
    "- https://plotly.com/python/interactive-html-export/",
]
check_inputs_unchanged()
with TemporaryDirectory(prefix=".eda-stage-", dir=OUT) as folder:
    stage = Path(folder)
    schemas = {}
    for filename, table in TABLES.items():
        table.to_csv(stage / filename, index=False, date_format="%Y-%m-%d", lineterminator="\n", encoding="utf-8")
        reload_csv_like(stage / filename, table)
        schemas[filename] = {"rows": len(table), "columns": table.columns.tolist(),
                             "dtypes": {c: str(table[c].dtype) for c in table}}
    (stage / "figures").mkdir()
    for filename, fig in FIGURES.items():
        fig.write_html(stage / "figures" / filename, include_plotlyjs=True, full_html=True,
                       config={"responsive": True, "displaylogo": False})
    (stage / "transport_rent_eda.md").write_text("\n".join(text_lines) + "\n", encoding="utf-8")
    files = [*TABLES, *("figures/" + n for n in FIGURES)]
    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "notebook": "08_transport_rent_eda.ipynb",
        "rental_quarter": RENT_QUARTER, "property_type": "2 bedroom flats", "boundary_year": 2025,
        "transport_service_date": SERVICE_DATE, "timezone": TIMEZONE, "am_window_seconds": [25200, 32400],
        "eligibility": spec["eligibility"], "feed_ids": spec["feed_ids"], "category_feed": CATEGORY_FEED,
        "input_sha256": {p.relative_to(ROOT).as_posix(): h for p, h in input_hashes.items()},
        "output_sha256": {n: fingerprint(stage/n) for n in files}, "output_schemas": schemas,
        "generated_report_sha256": fingerprint(stage / "transport_rent_eda.md"),
        "quality": {"service_grid_rows": len(lga_service), "comparison_lgas": len(combined),
                    "unmatched_rental_lgas": 0, "duplicate_codes": 0, "original_rental_fields_preserved": 14,
                    "am_events": AM_EVENTS, "distinct_study_area_am_trips": AM_UNIQUE_TRIPS,
                    "analytical_self_check_groups": len(self_checks), "accounting_checks_passed": True,
                    "csv_readback_matches": True, "raw_gtfs_reads_or_scans": 0, "input_files_unchanged": True},
        "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "plotly": plotly.__version__},
        "interpretation": "Differently dated snapshots; no score, ranking, prediction or causal inference. Stop-record ratios are not residential accessibility or headways.",
    }
    (stage / "eda_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    require(read_json(stage / "eda_manifest.json") == manifest, "Manifest read-back mismatch.")
    check_inputs_unchanged()
    for filename in files:
        destination = OUT / filename
        destination.parent.mkdir(parents=True, exist_ok=True)
        (stage / filename).replace(destination)
    (stage / "transport_rent_eda.md").replace(SUMMARY_PATH)
    (stage / "eda_manifest.json").replace(OUT / "eda_manifest.json")
check_inputs_unchanged()
print("TRANSPORT–RENT EDA CHECKPOINT PASSED")
print("Verified service indicator rows:", len(lga_service))
print("AM LGA/category rows:", len(am_categories))
print("Joined rental/transport LGAs:", len(combined))
print("Unmatched rental LGAs: 0 | Duplicate LGA codes: 0")
print("Original rental fields preserved: 14")
print("Rental quarter:", RENT_QUARTER, "| Transport service date:", SERVICE_DATE)
print("AM departure events reconciled:", AM_EVENTS)
print("Distinct study-area AM trips reconciled:", AM_UNIQUE_TRIPS)
print("Analytical self-check groups:", len(self_checks), "passed")
print("CSV tables saved and reloaded:", len(TABLES))
print("Interactive charts saved:", len(FIGURES))
print("Raw GTFS reads or scans performed: 0")
print("Input files unchanged: True")
print("Saved report:", SUMMARY_PATH.relative_to(ROOT))
print("Saved analysis:", OUT.relative_to(ROOT))


TRANSPORT–RENT EDA CHECKPOINT PASSED
Verified service indicator rows: 248
AM LGA/category rows: 124
Joined rental/transport LGAs: 31
Unmatched rental LGAs: 0 | Duplicate LGA codes: 0
Original rental fields preserved: 14
Rental quarter: 2025Q3 | Transport service date: 2026-10-13
AM departure events reconciled: 158932
Distinct study-area AM trips reconciled: 5768
Analytical self-check groups: 6 passed
CSV tables saved and reloaded: 6
Interactive charts saved: 4
Raw GTFS reads or scans performed: 0
Input files unchanged: True
Saved report: docs\transport_rent_eda.md
Saved analysis: data\interim\transport_rent_eda


## Your checkpoint and interpretation exercise

Save with **Ctrl+S** and open at least one exported HTML chart. The execution checkpoint verifies calculations/exports; your visual check confirms that the charts render and their labels are readable.

Explain one difference you notice between the charts in your own words. For example: "The LGA with more event records is not necessarily the one with the most distinct trips," or "Changing the stop-record denominator changes the mean without changing the event total." Use the actual charts rather than copying an expected finding.

Send the final checkpoint and one chart observation, or the first error. Do not repeat the large pipelines, install new packages, change expected hashes, or update recommendation scores to get past a problem.

The next product step will use this evidence to define transparent, appropriately limited comparison inputs. Raw LGA totals are not automatically suitable scoring features. No forecasts or recommendation claims have been validated here.